# 13. 최적화 알고리즘 — SGD에서 Adam까지

> **제13장** · **이론편 대응: 11.1절 (최적화 알고리즘)**
> **예상 소요**: 80분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음
> **다운로드**: 없음

---

## 이 장에서 하는 일

지금까지 학습에 쓴 것은 **기본 경사하강법**이었다.

$$w \leftarrow w - \eta \nabla L$$

간단하지만 문제가 많다. 이 장은 그 문제들과 해법을 다룬다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | **기본 경사하강법의 세 가지 문제** | 11.1절 |
| 2 | Momentum — 관성을 준다 | 11.1절 |
| 3 | RMSProp — 방향마다 다른 보폭 | 11.1절 |
| 4 | **Adam — 둘을 합치다** ★ | 11.1절 |
| 5 | **편향 보정 손계산 검증** ★ | 11.1절 |
| 6 | 실제 학습으로 비교 | 11.1절 |
| 7 | 학습률 스케줄링 | 11.5절 |
| 8 | 무엇을 쓸 것인가 | 11.1절 |

**4~5절이 핵심이다.** Adam이 왜 기본값이 되었는지,
그리고 편향 보정이 정확히 무슨 일을 하는지 손으로 확인한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=6, suppress=True)
print("준비 완료")

---

## 1. 기본 경사하강법의 세 가지 문제 — 이론편 11.1절

4장에서 선형회귀를 학습시킬 때 이미 하나를 겪었다 — **학습률을 잘못 잡으면 발산한다.**

문제는 그것만이 아니다.

| 문제 | 증상 |
|---|---|
| **1. 방향마다 기울기가 다름** | 좁은 골짜기에서 지그재그 |
| **2. 평평한 곳에서 멈춤** | 안장점·고원에서 진행이 없음 |
| **3. 학습률 하나로 전부** | 어떤 방향엔 크고 어떤 방향엔 작음 |

**첫 번째 문제부터 눈으로 확인한다.**

In [ ]:
import numpy as np


# 좁은 골짜기 함수 — 한 방향이 다른 방향보다 20배 가파르다
def f_valley(x):
    """f(x, y) = 0.5(x² + 20y²)"""
    return 0.5 * (x[0]**2 + 20 * x[1]**2)


def grad_valley(x):
    """기울기 = (x, 20y)"""
    return np.array([x[0], 20 * x[1]])


print("=" * 70)
print("좁은 골짜기 — 방향마다 기울기가 다르다")
print("=" * 70)
print("  f(x, y) = 0.5(x² + 20y²)")
print("  최솟값은 (0, 0)")
print()

test_points = [(5.0, 1.0), (5.0, 0.1), (1.0, 1.0)]
print(f"{'위치':<20}{'기울기':<26}{'비율 (y/x)'}")
print("-" * 70)
for p in test_points:
    g = grad_valley(np.array(p))
    ratio = abs(g[1] / g[0]) if g[0] != 0 else float("inf")
    print(f"{str(p):<20}{str(g.round(2)):<26}{ratio:.1f}배")
print("-" * 70)
print()
print("y 방향의 기울기가 훨씬 크다.")
print()
print("이것이 왜 문제인가")
print("  학습률을 y에 맞추면 → x 방향 진행이 너무 느리다")
print("  학습률을 x에 맞추면 → y 방향에서 튕겨 나간다")
print()
print("이론편 11.1절에서 '조건수가 크다'고 표현한 상황이다.")

In [ ]:
import numpy as np


def optimize(grad_fn, x0, method="SGD", lr=0.01, steps=100,
             beta1=0.9, beta2=0.999, eps=1e-8):
    """네 가지 최적화 알고리즘 (이론편 11.1절)

    같은 구조로 구현해 비교하기 쉽게 했다.
    """
    x = np.array(x0, dtype=float)
    trajectory = [x.copy()]

    m = np.zeros_like(x)      # 1차 모멘트 (기울기의 이동평균)
    v = np.zeros_like(x)      # 2차 모멘트 (기울기 제곱의 이동평균)

    for t in range(1, steps + 1):
        g = grad_fn(x)

        if method == "SGD":
            update = lr * g

        elif method == "Momentum":
            m = beta1 * m + g              # 속도에 누적
            update = lr * m

        elif method == "RMSProp":
            v = 0.9 * v + 0.1 * g**2       # 기울기 크기를 기억
            update = lr * g / (np.sqrt(v) + eps)

        elif method == "Adam":
            m = beta1 * m + (1 - beta1) * g
            v = beta2 * v + (1 - beta2) * g**2
            m_hat = m / (1 - beta1**t)     # 편향 보정 (5절)
            v_hat = v / (1 - beta2**t)
            update = lr * m_hat / (np.sqrt(v_hat) + eps)

        else:
            raise ValueError(f"알 수 없는 방법: {method}")

        x = x - update
        trajectory.append(x.copy())

    return np.array(trajectory)


print("=" * 70)
print("네 알고리즘의 갱신식")
print("=" * 70)
print()
print("  SGD       w ← w - η·g")
print()
print("  Momentum  m ← βm + g")
print("            w ← w - η·m")
print()
print("  RMSProp   v ← 0.9v + 0.1g²")
print("            w ← w - η·g / (√v + ε)")
print()
print("  Adam      m ← β₁m + (1-β₁)g        (Momentum 부분)")
print("            v ← β₂v + (1-β₂)g²       (RMSProp 부분)")
print("            m̂ = m/(1-β₁ᵗ),  v̂ = v/(1-β₂ᵗ)   (편향 보정)")
print("            w ← w - η·m̂ / (√v̂ + ε)")
print()
print("-" * 70)
print("Adam 은 Momentum 과 RMSProp 을 합친 것이다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("SGD 로 좁은 골짜기 내려가기")
print("=" * 70)

start = [5.0, 1.0]
lrs = [0.01, 0.05, 0.09, 0.11]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))

xx, yy = np.meshgrid(np.linspace(-6, 6, 200), np.linspace(-1.5, 1.5, 200))
zz = 0.5 * (xx**2 + 20 * yy**2)

for ax, lr in zip(axes, lrs):
    traj = optimize(grad_valley, start, "SGD", lr=lr, steps=60)

    ax.contour(xx, yy, zz, levels=25, colors="#94A3B8", linewidths=0.5)
    ax.plot(traj[:, 0], traj[:, 1], marker="o", markersize=2.5,
            linewidth=1.2, color="#DC2626")
    ax.plot(0, 0, marker="*", markersize=14, color="#0D9488")

    final = f_valley(traj[-1])
    diverged = not np.isfinite(final) or final > 1e6
    status = "발산" if diverged else f"f={final:.4f}"
    ax.set_title(f"lr={lr}  ({status})", fontsize=10)
    ax.set_xlim(-6, 6)
    ax.set_ylim(-1.5, 1.5)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print()
print(f"{'학습률':<12}{'60스텝 후 f':<20}{'상태'}")
print("-" * 70)
for lr in lrs:
    traj = optimize(grad_valley, start, "SGD", lr=lr, steps=60)
    final = f_valley(traj[-1])
    if not np.isfinite(final) or final > 1e6:
        print(f"{lr:<12}{'발산':<20}학습률이 너무 큼")
    else:
        print(f"{lr:<12}{final:<20.6f}{'느림' if final > 0.01 else '양호'}")
print("-" * 70)
print()
print("좁은 범위에서만 동작한다.")
print("  0.09 까지는 되고 0.11 이면 발산한다 — 안정 구간이 좁다")
print()
print("y 방향의 기울기 계수가 20 이므로, 학습률이 2/20 = 0.1 을 넘으면 발산한다.")
print("  4장에서 표준화 없이 학습했을 때와 같은 원리다.")

---

## 2. Momentum — 관성을 준다 — 이론편 11.1절

**공이 언덕을 굴러 내려가는 것**을 생각해 보자.
공은 매 순간 기울기 방향으로만 가는 것이 아니라 **이전 속도를 유지**한다.

$$m \leftarrow \beta m + g, \qquad w \leftarrow w - \eta m$$

$\beta = 0.9$면 이전 속도의 90%를 유지한다.

**효과 두 가지**

1. **지그재그 완화** — 왕복하는 방향은 서로 상쇄된다
2. **일관된 방향 가속** — 같은 방향이 반복되면 누적된다

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("SGD vs Momentum — 같은 조건에서")
print("=" * 70)

start = [5.0, 1.0]
lr = 0.04
steps = 60

traj_sgd = optimize(grad_valley, start, "SGD", lr=lr, steps=steps)
traj_mom = optimize(grad_valley, start, "Momentum", lr=lr * 0.2, steps=steps)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

xx, yy = np.meshgrid(np.linspace(-6, 6, 200), np.linspace(-1.5, 1.5, 200))
zz = 0.5 * (xx**2 + 20 * yy**2)

for ax, (name, traj, color) in zip(axes[:2], [
        ("SGD", traj_sgd, "#DC2626"),
        ("Momentum", traj_mom, "#0D9488")]):
    ax.contour(xx, yy, zz, levels=25, colors="#94A3B8", linewidths=0.5)
    ax.plot(traj[:, 0], traj[:, 1], marker="o", markersize=2.5,
            linewidth=1.2, color=color)
    ax.plot(0, 0, marker="*", markersize=14, color="#1E40AF")
    ax.set_title(f"{name}  (f={f_valley(traj[-1]):.5f})", fontsize=10)
    ax.set_xlim(-6, 6)
    ax.set_ylim(-1.5, 1.5)
    ax.grid(alpha=0.2)

# 손실 곡선
ax = axes[2]
for name, traj, color in [("SGD", traj_sgd, "#DC2626"),
                          ("Momentum", traj_mom, "#0D9488")]:
    losses = [f_valley(p) for p in traj]
    ax.plot(losses, linewidth=2, color=color, label=name)
ax.set_yscale("log")
ax.set_xlabel("스텝")
ax.set_ylabel("손실 (로그)")
ax.set_title("수렴 속도")
ax.legend()
ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print()
print("y 방향(세로)의 진동이 어떻게 다른지 보라.")
print()
print("[왜 진동이 줄어드나]")
print("  y 방향은 위아래로 왕복한다 → 기울기 부호가 계속 바뀐다")
print("  m = 0.9m + g 에서 부호가 반대인 값들이 서로 상쇄된다")
print()
print("  x 방향은 계속 같은 쪽이다 → 누적되어 가속된다")

In [ ]:
import numpy as np

print("=" * 70)
print("모멘텀 누적을 숫자로")
print("=" * 70)
print()
print("[경우 1] 같은 방향이 반복될 때 (기울기가 계속 0.1)")
m = 0.0
beta = 0.9
print(f"{'스텝':<8}{'기울기':<12}{'모멘텀 m':<14}{'배수'}")
print("-" * 70)
for t in range(1, 9):
    g = 0.1
    m = beta * m + g
    print(f"{t:<8}{g:<12.4f}{m:<14.4f}{m/g:.2f}배")
print("-" * 70)
print(f"수렴값: g/(1-β) = 0.1/0.1 = {0.1/(1-beta):.1f}  → 최대 10배 가속")
print()

print("[경우 2] 방향이 계속 바뀔 때 (기울기가 +0.1, -0.1 반복)")
m = 0.0
print(f"{'스텝':<8}{'기울기':<12}{'모멘텀 m':<14}{'상쇄 효과'}")
print("-" * 70)
for t in range(1, 9):
    g = 0.1 * (1 if t % 2 == 1 else -1)
    m = beta * m + g
    print(f"{t:<8}{g:<+12.4f}{m:<+14.4f}{'진폭 감소' if abs(m) < 0.1 else ''}")
print("-" * 70)
print()
print("일관된 방향은 최대 10배 가속되고, 왕복하는 방향은 억제된다.")
print("  이것이 모멘텀의 핵심이다.")

---

## 3. RMSProp — 방향마다 다른 보폭 — 이론편 11.1절

모멘텀은 **방향**을 다듬었다. RMSProp은 **보폭**을 다듬는다.

$$v \leftarrow 0.9v + 0.1g^2, \qquad w \leftarrow w - \frac{\eta}{\sqrt{v}+\epsilon}g$$

**기울기가 컸던 방향은 보폭을 줄이고, 작았던 방향은 늘린다.**

$g$를 $\sqrt{v}$로 나누므로, 기울기의 **크기**가 상쇄되고 **방향**만 남는 셈이다.

In [ ]:
import numpy as np

print("=" * 78)
print("RMSProp — 실효 학습률이 방향마다 달라진다")
print("=" * 78)

x = np.array([5.0, 1.0])
v = np.zeros(2)
lr = 0.1

print(f"{'스텝':<8}{'기울기 (x, y)':<26}{'√v (x, y)':<24}{'실효 학습률 (x, y)'}")
print("-" * 78)

for t in range(1, 9):
    g = grad_valley(x)
    v = 0.9 * v + 0.1 * g**2
    eff_lr = lr / (np.sqrt(v) + 1e-8)
    x = x - eff_lr * g

    if t <= 6 or t == 8:
        print(f"{t:<8}{str(g.round(3)):<26}{str(np.sqrt(v).round(3)):<24}"
              f"{str(eff_lr.round(4))}")

print("-" * 78)
print()
print("y 방향의 실효 학습률이 x 방향보다 훨씬 작다.")
print("  기울기가 컸던 방향이므로 자동으로 보폭이 줄었다.")
print()
print("[이것이 뜻하는 것]")
print("  학습률 하나로 모든 방향을 다루던 문제(1절)가 해결된다.")
print("  각 파라미터가 자기 사정에 맞는 보폭을 갖는다.")
print()
print("[주의] AdaGrad 와의 차이")
print("  AdaGrad: v 를 계속 더하기만 함 → 학습률이 0으로 수렴해 멈춤")
print("  RMSProp: 이동평균(0.9v + 0.1g²) → 오래된 것은 잊는다")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- 왼쪽: AdaGrad vs RMSProp 의 v 누적 ---
ax = axes[0]
steps = 100
g_const = 0.5

v_ada, v_rms = 0.0, 0.0
ada_lrs, rms_lrs = [], []
for t in range(steps):
    v_ada += g_const**2                    # 계속 누적
    v_rms = 0.9 * v_rms + 0.1 * g_const**2  # 이동평균
    ada_lrs.append(0.1 / (np.sqrt(v_ada) + 1e-8))
    rms_lrs.append(0.1 / (np.sqrt(v_rms) + 1e-8))

ax.plot(ada_lrs, linewidth=2.5, color="#DC2626", label="AdaGrad")
ax.plot(rms_lrs, linewidth=2.5, color="#0D9488", label="RMSProp")
ax.set_xlabel("스텝")
ax.set_ylabel("실효 학습률")
ax.set_title("AdaGrad 는 학습률이 0으로 수렴한다")
ax.legend()
ax.grid(alpha=0.3)

# --- 오른쪽: 궤적 비교 ---
ax = axes[1]
xx, yy = np.meshgrid(np.linspace(-6, 6, 200), np.linspace(-1.5, 1.5, 200))
zz = 0.5 * (xx**2 + 20 * yy**2)
ax.contour(xx, yy, zz, levels=25, colors="#94A3B8", linewidths=0.5)

for name, method, lr, color in [
        ("SGD", "SGD", 0.04, "#DC2626"),
        ("Momentum", "Momentum", 0.008, "#EA580C"),
        ("RMSProp", "RMSProp", 0.1, "#0D9488")]:
    traj = optimize(grad_valley, [5.0, 1.0], method, lr=lr, steps=60)
    ax.plot(traj[:, 0], traj[:, 1], marker="o", markersize=2,
            linewidth=1.2, color=color, label=name, alpha=0.85)

ax.plot(0, 0, marker="*", markersize=14, color="#1E40AF")
ax.set_xlim(-6, 6)
ax.set_ylim(-1.5, 1.5)
ax.set_title("세 방법의 궤적")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("RMSProp 이 골짜기를 따라 곧장 내려간다.")
print("  y 방향 보폭이 줄어 진동이 억제되고, x 방향은 상대적으로 커졌기 때문이다.")

---

## 4. Adam — 둘을 합치다 ★ — 이론편 11.1절

**Momentum(방향) + RMSProp(보폭) = Adam**

$$m \leftarrow \beta_1 m + (1-\beta_1)g \qquad \text{(방향)}$$
$$v \leftarrow \beta_2 v + (1-\beta_2)g^2 \qquad \text{(보폭)}$$
$$\hat{m} = \frac{m}{1-\beta_1^t}, \quad \hat{v} = \frac{v}{1-\beta_2^t} \qquad \text{(편향 보정)}$$
$$w \leftarrow w - \eta\frac{\hat{m}}{\sqrt{\hat{v}}+\epsilon}$$

**편향 보정이 왜 필요한가**가 5절의 주제다. 먼저 성능부터 보자.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("네 알고리즘 비교 — 좁은 골짜기")
print("=" * 78)

start = [5.0, 1.0]
steps = 80

# 각 방법에 맞는 학습률 (실무에서도 방법마다 다르게 잡는다)
configs = [
    ("SGD", 0.05, "#DC2626"),
    ("Momentum", 0.01, "#EA580C"),
    ("RMSProp", 0.1, "#94A3B8"),
    ("Adam", 0.3, "#0D9488"),
]

print(f"{'방법':<14}{'학습률':<12}{'최종 손실':<18}{'원점까지 거리'}")
print("-" * 78)

trajs = {}
for name, lr, color in configs:
    traj = optimize(grad_valley, start, name, lr=lr, steps=steps)
    trajs[name] = (traj, color)
    final_loss = f_valley(traj[-1])
    dist = np.linalg.norm(traj[-1])
    print(f"{name:<14}{lr:<12}{final_loss:<18.6f}{dist:.4f}")

print("-" * 78)
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- 왼쪽: 궤적 ---
ax = axes[0]
xx, yy = np.meshgrid(np.linspace(-6, 6, 200), np.linspace(-1.5, 1.5, 200))
zz = 0.5 * (xx**2 + 20 * yy**2)
ax.contour(xx, yy, zz, levels=25, colors="#CBD5E1", linewidths=0.5)

for name, (traj, color) in trajs.items():
    ax.plot(traj[:, 0], traj[:, 1], marker="o", markersize=2,
            linewidth=1.3, color=color, label=name, alpha=0.85)
ax.plot(0, 0, marker="*", markersize=15, color="#1E40AF")
ax.set_xlim(-6, 6)
ax.set_ylim(-1.5, 1.5)
ax.set_title("궤적 비교")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# --- 오른쪽: 손실 곡선 ---
ax = axes[1]
for name, (traj, color) in trajs.items():
    losses = [max(f_valley(p), 1e-12) for p in traj]
    ax.plot(losses, linewidth=2, color=color, label=name)
ax.set_yscale("log")
ax.set_xlabel("스텝")
ax.set_ylabel("손실 (로그)")
ax.set_title("수렴 속도")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print("[읽는 법]")
print("  각 방법에 맞는 학습률을 따로 잡았다는 점에 주의하자.")
print("  같은 학습률로 비교하면 공정하지 않다 — 갱신식의 크기가 다르기 때문")
print()
print("  Adam 은 lr=0.3 처럼 큰 값도 안정적이다.")
print("  √v 로 나누므로 기울기 크기와 무관하게 보폭이 결정되기 때문이다.")

---

## 5. 편향 보정 손계산 ★ — 이론편 11.1절

**Adam에서 가장 이해하기 어려운 부분**이다. 왜 $1-\beta^t$로 나눌까.

$m$과 $v$를 0에서 시작하기 때문이다. 첫 스텝에서

$$m_1 = 0.9 \times 0 + 0.1 \times g = 0.1g$$

**실제 기울기의 10%밖에 안 된다.** 이대로 쓰면 초반 갱신이 지나치게 작다.

보정하면

$$\hat{m}_1 = \frac{0.1g}{1 - 0.9^1} = \frac{0.1g}{0.1} = g$$

**정확히 $g$가 된다.** 이론편 11.1절에서 계산한 것을 확인해 보자.

In [ ]:
import numpy as np

print("=" * 78)
print("편향 보정 — 이론편 11.1절 값 검증")
print("=" * 78)
print()
print("설정: 기울기 g = 0.1 로 일정, β₁=0.9, β₂=0.999, η=0.001")
print()

g = 0.1
beta1, beta2, eps, lr = 0.9, 0.999, 1e-8, 0.001
m, v = 0.0, 0.0

print(f"{'t':<6}{'m':<14}{'m̂ (보정)':<14}{'v':<16}{'v̂ (보정)':<14}{'갱신량'}")
print("-" * 78)

updates_corrected = []
for t in range(1, 7):
    m = beta1 * m + (1 - beta1) * g
    v = beta2 * v + (1 - beta2) * g**2
    m_hat = m / (1 - beta1**t)
    v_hat = v / (1 - beta2**t)
    update = lr * m_hat / (np.sqrt(v_hat) + eps)
    updates_corrected.append(update)
    print(f"{t:<6}{m:<14.6f}{m_hat:<14.6f}{v:<16.8f}{v_hat:<14.8f}{update:.8f}")

print("-" * 78)
print()

# 검증
assert abs(updates_corrected[0] - lr) < 1e-6
print(f"[OK] t=1 에서 갱신량이 정확히 학습률({lr})과 같다")
print()
print("[왜 이렇게 되나]")
print("  기울기가 일정하면 m̂ = g, √v̂ = |g| 이므로")
print("  갱신량 = η·g/|g| = η")
print()
print("  즉 **기울기 크기와 무관하게 학습률만큼 움직인다.**")
print("  이것이 Adam 이 학습률 조정에 덜 민감한 이유다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("보정이 없으면 어떻게 되나")
print("=" * 78)

g = 0.1
m_c, v_c = 0.0, 0.0      # 보정 있음
m_n, v_n = 0.0, 0.0      # 보정 없음

corrected, no_correction = [], []

for t in range(1, 201):
    m_c = beta1 * m_c + (1 - beta1) * g
    v_c = beta2 * v_c + (1 - beta2) * g**2
    m_hat = m_c / (1 - beta1**t)
    v_hat = v_c / (1 - beta2**t)
    corrected.append(lr * m_hat / (np.sqrt(v_hat) + eps))

    m_n = beta1 * m_n + (1 - beta1) * g
    v_n = beta2 * v_n + (1 - beta2) * g**2
    no_correction.append(lr * m_n / (np.sqrt(v_n) + eps))

print(f"{'스텝':<10}{'보정 있음':<18}{'보정 없음':<18}{'비율'}")
print("-" * 78)
for t in [1, 2, 5, 10, 50, 100, 200]:
    c, n = corrected[t-1], no_correction[t-1]
    print(f"{t:<10}{c:<18.8f}{n:<18.8f}{n/c:.4f}")
print("-" * 78)
print()
print("보정이 없으면 초반 갱신이 지나치게 크다.")
print("  m 은 작지만 √v 가 더 작아서(β₂=0.999 라 훨씬 천천히 커짐)")
print("  나눈 결과가 오히려 커진다.")
print()
print("  → 학습 초반에 파라미터가 크게 튀어 불안정해진다")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

ax = axes[0]
ax.plot(range(1, 61), corrected[:60], linewidth=2.5,
        color="#0D9488", label="보정 있음")
ax.plot(range(1, 61), no_correction[:60], linewidth=2.5,
        color="#DC2626", label="보정 없음")
ax.axhline(lr, color="#1E40AF", linestyle="--", linewidth=1.5)
ax.text(30, lr * 1.15, f"학습률 {lr}", fontsize=8, color="#1E40AF")
ax.set_xlabel("스텝")
ax.set_ylabel("갱신량")
ax.set_title("초반 갱신량 비교")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[1]
ts = np.arange(1, 201)
ax.plot(ts, 1 - beta1**ts, linewidth=2.5, color="#EA580C", label="1 - beta1^t (0.9)")
ax.plot(ts, 1 - beta2**ts, linewidth=2.5, color="#1E40AF", label="1 - beta2^t (0.999)")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("스텝 t")
ax.set_ylabel("보정 계수")
ax.set_title("보정 계수는 1로 수렴한다")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("오른쪽 그래프의 요지")
print("  t 가 커지면 1-βᵗ → 1 이므로 보정 효과가 사라진다.")
print("  즉 편향 보정은 **초반에만** 작동한다.")
print()
print(f"  β₁=0.9  : {int(np.argmax(1-beta1**ts > 0.99))+1}스텝이면 99% 도달")
print(f"  β₂=0.999: {int(np.argmax(1-beta2**ts > 0.99))+1}스텝이면 99% 도달")

---

## 6. 실제 학습으로 비교 — 이론편 11.1절

수학 함수가 아니라 **실제 신경망 학습**에서 비교한다.
11장에서 만든 MLP를 다시 쓴다.

In [ ]:
import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))


class SimpleMLP:
    """작은 MLP — 11장에서 만든 것과 같은 구조"""

    def __init__(self, n_in, n_hidden, n_out, seed=42):
        rng = np.random.RandomState(seed)
        self.W1 = rng.randn(n_in, n_hidden) * np.sqrt(2.0 / n_in)
        self.b1 = np.zeros(n_hidden)
        self.W2 = rng.randn(n_hidden, n_out) * np.sqrt(2.0 / n_hidden)
        self.b2 = np.zeros(n_out)

    def params(self):
        return [self.W1, self.b1, self.W2, self.b2]

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = np.maximum(0, self.z1)          # ReLU
        self.z2 = self.a1 @ self.W2 + self.b2
        self.out = sigmoid(self.z2)
        return self.out

    def backward(self, X, y):
        n = len(X)
        d2 = (self.out - y.reshape(-1, 1)) / n
        gW2 = self.a1.T @ d2
        gb2 = d2.sum(axis=0)
        d1 = (d2 @ self.W2.T) * (self.z1 > 0)
        gW1 = X.T @ d1
        gb1 = d1.sum(axis=0)
        return [gW1, gb1, gW2, gb2]

    def loss(self, X, y):
        p = np.clip(self.forward(X), 1e-9, 1 - 1e-9)
        y = y.reshape(-1, 1)
        return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def train(method, X, y, lr, epochs=200, seed=42):
    """지정한 최적화 방법으로 학습한다"""
    model = SimpleMLP(X.shape[1], 16, 1, seed=seed)
    params = model.params()

    ms = [np.zeros_like(p) for p in params]
    vs = [np.zeros_like(p) for p in params]

    losses = []
    for t in range(1, epochs + 1):
        model.forward(X)
        grads = model.backward(X, y)

        for i, (p, g) in enumerate(zip(params, grads)):
            if method == "SGD":
                p -= lr * g
            elif method == "Momentum":
                ms[i] = 0.9 * ms[i] + g
                p -= lr * ms[i]
            elif method == "RMSProp":
                vs[i] = 0.9 * vs[i] + 0.1 * g**2
                p -= lr * g / (np.sqrt(vs[i]) + 1e-8)
            elif method == "Adam":
                ms[i] = 0.9 * ms[i] + 0.1 * g
                vs[i] = 0.999 * vs[i] + 0.001 * g**2
                m_hat = ms[i] / (1 - 0.9**t)
                v_hat = vs[i] / (1 - 0.999**t)
                p -= lr * m_hat / (np.sqrt(v_hat) + 1e-8)

        losses.append(model.loss(X, y))

    return model, losses


# 데이터 생성
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

X_data, y_data = make_moons(n_samples=400, noise=0.25, random_state=42)
X_data = StandardScaler().fit_transform(X_data)

print("=" * 78)
print("실제 학습 비교 — make_moons 분류")
print("=" * 78)
print(f"데이터: {X_data.shape[0]}개, 특성 {X_data.shape[1]}개")
print()

settings = [
    ("SGD", 0.5),
    ("Momentum", 0.05),
    ("RMSProp", 0.01),
    ("Adam", 0.05),
]

results = {}
print(f"{'방법':<14}{'학습률':<12}{'최종 손실':<16}{'정확도'}")
print("-" * 78)
for method, lr in settings:
    model, losses = train(method, X_data, y_data, lr, epochs=300)
    pred = (model.forward(X_data) > 0.5).astype(int).ravel()
    acc = (pred == y_data).mean()
    results[method] = losses
    print(f"{method:<14}{lr:<12}{losses[-1]:<16.6f}{acc:.4f}")
print("-" * 78)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

colors = {"SGD": "#DC2626", "Momentum": "#EA580C",
          "RMSProp": "#94A3B8", "Adam": "#0D9488"}

# --- 왼쪽: 손실 곡선 ---
ax = axes[0]
for method, losses in results.items():
    ax.plot(losses, linewidth=2, color=colors[method], label=method)
ax.set_xlabel("에폭")
ax.set_ylabel("손실")
ax.set_title("학습 곡선")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- 오른쪽: 초반 확대 ---
ax = axes[1]
for method, losses in results.items():
    ax.plot(losses[:60], linewidth=2, color=colors[method], label=method)
ax.set_xlabel("에폭")
ax.set_ylabel("손실")
ax.set_title("초반 60에폭 확대")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 78)
print("몇 에폭 만에 손실 0.3 이하로 내려가나")
print("=" * 78)
print(f"{'방법':<14}{'도달 에폭':<16}{'비고'}")
print("-" * 78)
for method, losses in results.items():
    reached = next((i+1 for i, l in enumerate(losses) if l < 0.3), None)
    note = "" if reached else "300에폭 내 미도달"
    print(f"{method:<14}{str(reached) if reached else '—':<16}{note}")
print("-" * 78)
print()
print("[주의] 이 비교의 한계")
print("  학습률을 각 방법에 맞게 조정했으므로 '누가 빠른가'는 조건에 따라 달라진다.")
print("  실무의 요점은 **Adam 이 학습률 조정에 덜 민감하다**는 것이다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("학습률 민감도 — 이것이 Adam 이 기본값인 이유")
print("=" * 78)

lr_range = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]
sensitivity = {}

print(f"{'학습률':<12}" + "".join(f"{m:<14}" for m in ["SGD", "Momentum", "Adam"]))
print("-" * 78)

for lr in lr_range:
    row = f"{lr:<12}"
    for method in ["SGD", "Momentum", "Adam"]:
        try:
            _, losses = train(method, X_data, y_data, lr, epochs=150)
            final = losses[-1]
            if not np.isfinite(final) or final > 10:
                cell = "발산"
            else:
                cell = f"{final:.4f}"
        except Exception:
            cell = "발산"
        sensitivity.setdefault(method, []).append(
            final if np.isfinite(final) and final < 10 else np.nan)
        row += f"{cell:<14}"
    print(row)

print("-" * 78)
print()

fig, ax = plt.subplots(figsize=(8.5, 4.2))
for method, color in [("SGD", "#DC2626"), ("Momentum", "#EA580C"),
                      ("Adam", "#0D9488")]:
    vals = sensitivity[method]
    ax.plot(lr_range, vals, marker="o", linewidth=2.5,
            color=color, label=method)

ax.set_xscale("log")
ax.set_xlabel("학습률 (로그)")
ax.set_ylabel("최종 손실")
ax.set_title("학습률에 따른 최종 손실 (빈 구간 = 발산)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print("Adam 이 넓은 학습률 범위에서 안정적으로 동작한다.")
print()
print("  실무에서 'Adam, lr=0.001' 로 시작하는 것이 관행이 된 이유다.")
print("  일단 돌아가게 만든 뒤 세밀하게 조정하면 된다.")

---

## 7. 학습률 스케줄링 — 이론편 11.5절

Adam을 써도 **학습률 자체를 조절하면** 더 나아진다.

**발상**: 처음에는 크게 움직이고, 나중에는 미세 조정한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def lr_step(base, epoch, drop=0.5, every=50):
    """계단식 감소"""
    return base * (drop ** (epoch // every))


def lr_exponential(base, epoch, gamma=0.99):
    """지수 감소"""
    return base * (gamma ** epoch)


def lr_cosine(base, epoch, total=200, min_lr=0.0):
    """코사인 감소 — 최근 많이 쓰인다"""
    return min_lr + (base - min_lr) * 0.5 * (1 + np.cos(np.pi * epoch / total))


def lr_warmup_cosine(base, epoch, warmup=20, total=200):
    """워밍업 + 코사인 (Transformer 학습의 표준)"""
    if epoch < warmup:
        return base * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(total - warmup, 1)
    return base * 0.5 * (1 + np.cos(np.pi * progress))


print("=" * 70)
print("학습률 스케줄 종류 (이론편 11.5절)")
print("=" * 70)
print()
print(f"{'방식':<20}{'특징':<30}{'주로 쓰이는 곳'}")
print("-" * 70)
print(f"{'고정':<20}{'단순, 조정 불필요':<30}{'간단한 문제'}")
print(f"{'계단식':<20}{'정해진 시점에 감소':<30}{'전통적인 CNN 학습'}")
print(f"{'지수':<20}{'매 스텝 조금씩 감소':<30}{'일반적'}")
print(f"{'코사인':<20}{'부드럽게 0까지':<30}{'최근 표준'}")
print(f"{'워밍업+코사인':<20}{'초반에 천천히 올림':<30}{'Transformer (18장)'}")
print("-" * 70)

epochs = 200
base_lr = 0.01

fig, ax = plt.subplots(figsize=(9, 4.2))
for name, fn, color in [
        ("고정", lambda e: base_lr, "#94A3B8"),
        ("계단식", lambda e: lr_step(base_lr, e), "#DC2626"),
        ("지수", lambda e: lr_exponential(base_lr, e), "#EA580C"),
        ("코사인", lambda e: lr_cosine(base_lr, e, epochs), "#0D9488"),
        ("워밍업+코사인", lambda e: lr_warmup_cosine(base_lr, e, 20, epochs), "#1E40AF")]:
    values = [fn(e) for e in range(epochs)]
    ax.plot(values, linewidth=2.2, color=color, label=name)

ax.set_xlabel("에폭")
ax.set_ylabel("학습률")
ax.set_title("스케줄 비교")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print()
print("[워밍업이 필요한 이유]")
print("  학습 초반에는 파라미터가 무작위라 기울기가 불안정하다.")
print("  이때 큰 학습률을 쓰면 엉뚱한 방향으로 크게 움직인다.")
print()
print("  18장 Transformer 학습에서 워밍업이 사실상 필수인 이유다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def train_with_schedule(schedule_fn, X, y, base_lr, epochs=200, seed=42):
    """스케줄을 적용해 Adam 으로 학습"""
    model = SimpleMLP(X.shape[1], 16, 1, seed=seed)
    params = model.params()
    ms = [np.zeros_like(p) for p in params]
    vs = [np.zeros_like(p) for p in params]

    losses, lrs = [], []
    for t in range(1, epochs + 1):
        lr = schedule_fn(t - 1)
        lrs.append(lr)

        model.forward(X)
        grads = model.backward(X, y)
        for i, (p, g) in enumerate(zip(params, grads)):
            ms[i] = 0.9 * ms[i] + 0.1 * g
            vs[i] = 0.999 * vs[i] + 0.001 * g**2
            m_hat = ms[i] / (1 - 0.9**t)
            v_hat = vs[i] / (1 - 0.999**t)
            p -= lr * m_hat / (np.sqrt(v_hat) + 1e-8)

        losses.append(model.loss(X, y))

    return losses, lrs


print("=" * 78)
print("스케줄 적용 효과")
print("=" * 78)

epochs = 250
base_lr = 0.05

schedules = {
    "고정": lambda e: base_lr,
    "계단식": lambda e: lr_step(base_lr, e, 0.5, 60),
    "코사인": lambda e: lr_cosine(base_lr, e, epochs),
    "워밍업+코사인": lambda e: lr_warmup_cosine(base_lr, e, 25, epochs),
}

sched_results = {}
print(f"{'스케줄':<20}{'최종 손실':<18}{'최저 손실':<18}{'정확도'}")
print("-" * 78)
for name, fn in schedules.items():
    losses, lrs = train_with_schedule(fn, X_data, y_data, base_lr, epochs)
    sched_results[name] = (losses, lrs)
    model = SimpleMLP(X_data.shape[1], 16, 1, seed=42)
    print(f"{name:<20}{losses[-1]:<18.6f}{min(losses):<18.6f}", end="")
    # 정확도 재계산
    m2, l2 = train_with_schedule(fn, X_data, y_data, base_lr, epochs)
    print(f"{'':>6}")
print("-" * 78)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
colors_s = ["#94A3B8", "#DC2626", "#0D9488", "#1E40AF"]

ax = axes[0]
for (name, (losses, _)), color in zip(sched_results.items(), colors_s):
    ax.plot(losses, linewidth=2, color=color, label=name)
ax.set_xlabel("에폭")
ax.set_ylabel("손실")
ax.set_title("스케줄에 따른 학습 곡선")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[1]
for (name, (losses, _)), color in zip(sched_results.items(), colors_s):
    ax.plot(losses[-80:], linewidth=2, color=color, label=name)
ax.set_xlabel("마지막 80에폭")
ax.set_ylabel("손실")
ax.set_title("후반부 확대 — 미세 조정 구간")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("후반부를 보면 스케줄을 쓴 쪽이 더 낮고 안정적이다.")
print("  학습률이 줄어 손실 표면의 좁은 골짜기 바닥에 안착하기 때문이다.")

---

## 8. 무엇을 쓸 것인가 — 이론편 11.1절

정리하면 이렇다.

In [ ]:
print("=" * 78)
print("선택 기준")
print("=" * 78)
print()
print(f"{'상황':<32}{'권장':<22}{'이유'}")
print("-" * 78)
choices = [
    ("처음 시작할 때",          "Adam (lr=0.001)",    "학습률에 덜 민감"),
    ("이미지 분류 (CNN)",       "SGD + Momentum",     "최종 성능이 더 좋다는 보고"),
    ("Transformer 학습",       "AdamW + 워밍업",      "18장에서 다룬 구조"),
    ("파인튜닝 (25장)",         "AdamW (작은 lr)",     "원본에서 크게 벗어나지 않게"),
    ("메모리가 부족할 때",       "SGD",                "상태를 저장하지 않음"),
    ("재현성이 중요할 때",       "SGD",                "동작이 단순"),
]
for a, b, c in choices:
    print(f"{a:<32}{b:<22}{c}")
print("-" * 78)
print()

print("[AdamW 는 무엇인가]")
print()
print("  Adam 에 가중치 감쇠(weight decay)를 올바르게 적용한 변형이다.")
print("  Adam 에서 L2 정규화를 그냥 넣으면 √v 로 나뉘어 효과가 왜곡된다.")
print("  AdamW 는 감쇠를 갱신식에서 분리해 적용한다.")
print()
print("  현재 대부분의 LLM 학습이 AdamW 를 쓴다.")
print()

print("[메모리 관점 — 24장 6절과 연결]")
print()
print(f"{'옵티마이저':<20}{'파라미터당 저장':<24}{'모델 대비'}")
print("-" * 78)
print(f"{'SGD':<20}{'없음':<24}{'x1 (가중치만)'}")
print(f"{'Momentum':<20}{'m':<24}{'x2'}")
print(f"{'RMSProp':<20}{'v':<24}{'x2'}")
print(f"{'Adam / AdamW':<20}{'m, v':<24}{'x3'}")
print("-" * 78)
print()
print("24장에서 '학습에 가중치의 4배 메모리'라고 한 것이 이것이다.")
print("  가중치 1 + 그래디언트 1 + Adam 상태 2 = 4")
print()
print("  → 26장의 LoRA 가 이 부담을 줄이는 방법이다.")

In [ ]:
print("=" * 78)
print("PyTorch 에서 쓰는 법")
print("=" * 78)
print()
print("13장에서 PyTorch 를 다룰 때 이미 만나게 된다.")
print()
example = [
    "import torch.optim as optim",
    "",
    "# 기본 선택",
    "optimizer = optim.Adam(model.parameters(), lr=0.001)",
    "",
    "# 파인튜닝에는 AdamW",
    # ── torch.optim.AdamW 파라미터 ───────────────────────────────
    #   Adam 과 같으나 weight_decay 처리 방식이 다르다.
    #   Adam 은 L2 항이 적응적 학습률에 나뉘어 효과가 왜곡되는데,
    #   AdamW 는 갱신식에서 분리해 적용한다 → **LLM 학습의 표준**
    #
    #   lr            기본값 0.001.  파인튜닝은 1e-5 ~ 5e-5
    #   weight_decay  기본값 0.01 (Adam 의 0 과 다름)
    #                 예: 0.01(기본) / 0.1(강한 정규화)
    #   betas, eps    Adam 과 동일
    # ──────────────────────────────────────────────────────────────
    "optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)",
    "",
    "# 전통적인 CNN 학습",
    "optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)",
    "",
    "# 스케줄러 함께 쓰기",
    # ── 학습률 스케줄러 (torch.optim.lr_scheduler) ───────────────
    #   StepLR(optimizer, step_size=30, gamma=0.1)
    #       step_size 에폭마다 lr 에 gamma 를 곱한다
    #   CosineAnnealingLR(optimizer, T_max=200, eta_min=0)
    #       T_max 에폭에 걸쳐 코사인 곡선으로 eta_min 까지 감소
    #   ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)
    #       지표가 patience 에폭 동안 개선되지 않으면 factor 를 곱한다
    #   OneCycleLR(optimizer, max_lr=0.1, total_steps=...)
    #       올렸다가 내리는 방식. **배치마다** step() 을 부른다
    #
    #   [주의] 대부분 에폭마다 scheduler.step() 을 부른다.
    #          OneCycleLR 등 일부는 배치마다 부른다 — 문서 확인
    # ──────────────────────────────────────────────────────────────
    "scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)",
    "",
    "for epoch in range(200):",
    "    for batch in loader:",
    "        optimizer.zero_grad()",
    "        loss = criterion(model(batch.x), batch.y)",
    "        loss.backward()",
    "        optimizer.step()",
    "    scheduler.step()          # 에폭마다 학습률 갱신",
]
for line in example:
    print("  " + line)

print()
print("-" * 78)
print("[주의] scheduler.step() 위치")
print("  대부분의 스케줄러는 **에폭마다** 부른다.")
print("  일부(OneCycleLR 등)는 배치마다 부른다 — 문서 확인 필요")

---

## 9. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| **11.1** | **Adam 편향 보정 후 t=1 갱신 = 학습률** | **검증** ✓ |
| 11.1 | 모멘텀 최대 가속 = 1/(1−β) = 10배 | 계산 ✓ |
| 11.1 | AdaGrad는 학습률이 0으로 수렴 | 그래프 확인 ✓ |
| 11.1 | Adam이 학습률에 덜 민감 | 실험 확인 ✓ |
| 11.5 | 워밍업의 필요성 | 스케줄 비교 ✓ |

### 네 알고리즘 요약

| 방법 | 무엇을 다듬나 | 핵심 아이디어 |
|---|---|---|
| SGD | — | 기울기 방향으로 그냥 |
| Momentum | **방향** | 이전 속도를 유지 |
| RMSProp | **보폭** | 기울기가 컸던 방향은 줄임 |
| **Adam** | **둘 다** | Momentum + RMSProp + 편향 보정 |

### 기억할 것

| 항목 | 요점 |
|---|---|
| 좁은 골짜기 | 방향마다 기울기가 달라 지그재그 |
| 모멘텀 | 왕복 방향은 상쇄, 일관된 방향은 가속 |
| RMSProp | √v로 나눠 방향별 보폭 조정 |
| **편향 보정** | **초반 갱신이 작아지는 것을 보정** |
| Adam 기본값 | lr=0.001, β₁=0.9, β₂=0.999 |
| AdamW | 가중치 감쇠를 올바르게 — LLM 학습의 표준 |
| 옵티마이저 메모리 | Adam은 모델의 **3배** |
| 스케줄링 | 후반 미세 조정에 효과 |

### 다음 장

**14. 정규화 기법 — 과대적합과 싸우기** — 지금까지 손으로 구현한 것들이
PyTorch에서는 한 줄로 처리된다. `optim.Adam` 안에 이 장의 내용이 들어 있다.